# HyperOpt

https://hyperopt.github.io/hyperopt/

HyperOpt는 하이퍼파라미터 탐색을 더 효율적으로 수행하기 위한 최적화 라이브러리이다.

앞에서 Grid Search, Random Search처럼 여러 하이퍼파라미터 조합을 시도해 보는 방법을 보았다.
이제 HyperOpt에서는 모든 조합을 단순히 다 해보는 것이 아니라, 이전 시도의 결과를 바탕으로 다음 탐색 위치를 더 똑똑하게 정하는 방식에 주목한다.

1. 하이퍼파라미터 튜닝은 성능 향상에 중요하지만, 모든 조합을 전부 실험하는 것은 시간과 비용이 많이 든다.
2. HyperOpt는 이전 실험 결과를 바탕으로 다음 후보를 더 효율적으로 탐색하는 베이지안 최적화 계열 라이브러리이다.
3. 따라서 단순히 best parameter만 보는 것이 아니라, 탐색 공간을 어떻게 정의하는지와 반복 탐색이 어떤 방식으로 이루어지는지도 함께 이해하는 것이 중요하다.

## 왜 HyperOpt를 사용하는가?

Grid Search는 정해둔 후보를 전부 다 확인한다.
그래서 탐색 범위가 넓어질수록 경우의 수가 급격히 늘어난다.

Random Search는 후보를 무작위로 뽑아서 시도한다.
Grid Search보다 덜 비효율적일 수 있지만, 이전 결과를 바탕으로 다음 탐색을 조정하지는 않는다.

HyperOpt는 이전 실험 결과를 참고하여, 성능이 좋을 가능성이 높은 방향을 다음 탐색에 반영한다.
즉, 적은 시도로도 괜찮은 하이퍼파라미터 조합을 찾도록 돕는 것이 핵심이다.

## HyperOpt의 핵심 아이디어

1. 탐색할 하이퍼파라미터 범위를 먼저 정의한다.
2. 몇 번의 실험을 통해 각 조합의 성능을 확인한다.
3. 이전 실험 결과를 바탕으로, 다음에는 어디를 시도할지 더 유망한 후보를 선택한다.
4. 이 과정을 반복하면서 더 좋은 조합을 찾아간다.

즉, "지금까지의 결과를 보고 다음 시도를 조정한다"는 점이 HyperOpt의 핵심이다.

## HyperOpt에서 함께 봐야 할 것

1. search space
   * 어떤 하이퍼파라미터를 어떤 범위에서 탐색할지 정의한다.
   * 예를 들어 learning_rate는 0.01 ~ 0.3 사이, max_depth는 3 ~ 10 사이처럼 설정할 수 있다.

2. objective function
   * 주어진 하이퍼파라미터 조합이 얼마나 좋은지 점수로 반환하는 함수이다.
   * 보통 검증 데이터의 accuracy, f1-score, RMSE 같은 평가 지표를 사용한다.

3. trials
   * 각 탐색 결과를 기록하는 객체이다.
   * 어떤 조합을 시도했고, 그 결과가 어땠는지 추적할 수 있다.

4. fmin
   * HyperOpt에서 최적화를 실제로 수행하는 함수이다.
   * 정의한 탐색 공간과 objective function을 바탕으로 최적의 조합을 찾는다.

## HyperOpt의 장점

1. 효율적인 탐색
   * 모든 조합을 전부 시도하지 않아도 된다.
   * 이전 결과를 반영하여 더 가능성 있는 후보를 우선적으로 탐색한다.

2. 복잡한 탐색 공간 처리
   * 정수형, 실수형, 범주형 하이퍼파라미터를 함께 정의할 수 있다.
   * 조건부 탐색 공간도 비교적 유연하게 구성할 수 있다.

3. 실무 적용에 적합
   * 모델 학습 비용이 큰 경우, 제한된 횟수 안에서 합리적인 조합을 찾는 데 유리하다.
   * 특히 부스팅 계열 모델처럼 튜닝할 파라미터가 많은 경우 유용하다.

## 주의할 점

1. HyperOpt가 항상 최적해를 보장하는 것은 아니다.
   * 제한된 횟수 안에서 더 좋은 후보를 효율적으로 찾도록 돕는 방식에 가깝다.

2. 탐색 공간을 너무 넓게 잡으면 비효율적일 수 있다.
   * 어떤 파라미터를 어느 범위에서 탐색할지 적절히 설계하는 것이 중요하다.

3. objective function을 어떻게 정의하느냐가 중요하다.
   * 검증 방식이나 평가 지표가 부적절하면, 찾은 하이퍼파라미터도 좋은 결과로 이어지지 않을 수 있다.

## 환경설정
HyperOpt는 내부적으로 pkg_resources에 의존하는 버전 이슈가 있어, 일부 환경에서는 setuptools 버전을 낮춰야 정상 동작할 수 있다.

In [ ]:
%conda install hyperopt
%pip install "setuptools<82"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### iris + RandomForestClassfier

In [ ]:
# 데이터 준비
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())